In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import pandas_toon
import time
#from scrapper_functions  import *

#from sentence_transformers import SentenceTransformer
#import lancedb

In [8]:
def get_ingredients(soup):
    ingredients = soup.find_all(class_="ingredient")
    quantities = soup.find_all(class_="qty")
    #print(ingredients,"\n\n\n")
    result_ingredients = []
    if len(quantities) < len(ingredients):
        for i,child in enumerate(ingredients):
            for description in child.children:
                result_ingredients.append([description])
        return result_ingredients
    else:
        for i,child in enumerate(ingredients):
            for kind in child.children:
                for qty in quantities[i].children:
                    result_ingredients.append([kind,qty])
        return result_ingredients


def get_category(soup):
    cat = soup.find(class_="post-categories")
    return next(cat.a.children,None)


def get_title(soup):
    tit = soup.find('h1')
    return next(tit.children,None)

In [10]:
with open("links.csv","r") as f:
    links = f.readlines()
headers = {
    "User-Agent": "Mozilla/5.0"
}
count = 0
for i,link_raw in enumerate(links):
    try:
        link = link_raw.strip()
        page = requests.get(link, headers=headers)
        soup = BeautifulSoup(page.content)
        ingredients = get_ingredients(soup)
        title = get_title(soup)
        category = get_category(soup)
        df = pd.DataFrame({'url': [link],
                          'recipe_name': [title],
                           'ingredients': [ingredients]})
        df.to_json('data_ania.json', orient='records', lines=True, mode='a')
        time.sleep(2.5)
    except Exception as e:
        count+=1
        print(i)
        print(link_raw)
        print(e,"\n\n")
        continue

print(count)

2201
https://aniagotuje.pl/przepisy/zupy

'NoneType' object has no attribute 'a' 


2202
https://aniagotuje.pl/przepisy/dania-bez-miesa

'NoneType' object has no attribute 'a' 


2203
https://aniagotuje.pl/przepisy/dania-na-slodko

'NoneType' object has no attribute 'a' 


2204
https://aniagotuje.pl/przepisy/serniki

'NoneType' object has no attribute 'a' 


2205
https://aniagotuje.pl/przepisy/przetwory

'NoneType' object has no attribute 'a' 


2206
https://aniagotuje.pl/przepisy/ciasta-i-torty

'NoneType' object has no attribute 'a' 


2207
https://aniagotuje.pl/przepisy/lody-domowe

'NoneType' object has no attribute 'a' 


2208
https://aniagotuje.pl/przepisy/sosy-i-pasty

'NoneType' object has no attribute 'a' 


2209
https://aniagotuje.pl/przepisy/nalesniki-i-placki

'NoneType' object has no attribute 'a' 


2210
https://aniagotuje.pl/przepisy/dania-miesne

'NoneType' object has no attribute 'a' 


2211
https://aniagotuje.pl/przepisy/chleby-i-bulki

'NoneType' object has no attrib

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
db = lancedb.connect("recipe_base")

In [ ]:
for i in range(len(links)):
    try:
        df = pandas_toon.read_toon(f"data/proba{i}")
    except IOError as e:
        print(e)
        continue
    if "recipe_base" not in db.list_tables():
        # Jeśli to nasz pierwszy raz - tworzymy tabelę
        tabela = db.create_table("recipe_base", data=df)
        print("Utworzono nową bazę i dodano pierwszy przepis!")
    else:
        # Jeśli baza już istnieje na dysku - DODAJEMY nowe dane
        tabela = db.open_table("recipe_base")
        tabela.add(df)
        print("Dodano kolejne przepis do istniejącej bazy!")
    break